In [1]:
%load_ext autoreload
%autoreload 2

**Author:** Salvador Navas  
**Date:** 2025-06-27

In [2]:
from pyhydra.climate.spatial_analysis import (
    fit_gev_mle,
    fit_gev_lmom,
    fit_gev_bayes,
    return_level,
    return_level_bayes,
    regional_index_flood,
    fit_regional_gev,
    regional_return_levels,
)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Regional Frequency Analysis (GEV)

## What is Regional Frequency Analysis?

At-site frequency analysis fits a distribution to the record at a single gauge. For rare events (T > 50 yr), the uncertainty is often so large that the result is almost useless — a 40-year record can only directly observe ~40 data points, barely enough to fit a 3-parameter distribution.

**Regional Frequency Analysis (RFA)** solves this by pooling data from multiple stations in a **hydrologically homogeneous region**. The key insight: extreme event statistics (especially the shape parameter ξ) are similar across a region even when mean flows differ. By normalising each station, we can pool perhaps 200–300 station-years into a single regional estimate.

---

## The Index Flood method (Dalrymple 1960)

1. **Index flood** = each station's mean annual maximum (Q̄_s)
2. **Normalise**: divide each station's series by its own index flood → dimensionless "growth curve" series
3. **Pool**: concatenate all normalised series (now comparable between stations)
4. **Fit**: estimate one regional GEV on the pooled data
5. **Rescale**: multiply regional T-year quantile by each station's index flood to get site-specific return levels

The fundamental assumption: **all stations in the region share the same dimensionless GEV** (same ξ, same shape, different means).

---

## Fitting methods available

| Function | Method | Dependencies | Recommended for |
|----------|--------|--------------|----------------|
| `fit_gev_mle` | Maximum Likelihood | `scipy` | Samples ≥ 30 |
| `fit_gev_lmom` | L-moments | `lmoments3` | Any sample size, robust to outliers |
| `fit_gev_bayes` | Bayesian MCMC (PyMC) | `pymc` | Full uncertainty quantification |
| `return_level_bayes` | Posterior return levels | `pymc` | Credible intervals on return levels |

---

## Installation
```bash
pip install scipy lmoments3    # core methods
pip install pymc               # Bayesian method only
```

---
## Synthetic dataset

Eight stations representing major Iberian river basins, generated with:
- **Shared shape parameter** ξ ≈ 0.12–0.20: homogeneous region assumption
- **Different location/scale**: reflecting the index-flood structure (mean annual maximum varies by basin)
- **Different record lengths** (25–45 years): realistic heterogeneity

The synthetic data allows us to know the "true" parameters and validate the regional estimates.

> In a real study, the homogeneity assumption should be tested before pooling — see the **Hosking & Wallis heterogeneity check** in Section 3.

In [3]:
from scipy.stats import genextreme

rng = np.random.default_rng(42)

# Station characteristics
stations = {
    "Ebro":     {"mu": 1200, "sigma": 350, "xi": 0.15, "n": 40},
    "Tajo":     {"mu":  900, "sigma": 260, "xi": 0.12, "n": 35},
    "Duero":    {"mu": 1100, "sigma": 310, "xi": 0.14, "n": 45},
    "Guadalquivir": {"mu": 800, "sigma": 240, "xi": 0.10, "n": 38},
    "Turia":    {"mu":  350, "sigma": 120, "xi": 0.18, "n": 30},
    "Segura":   {"mu":  280, "sigma":  90, "xi": 0.20, "n": 25},
    "Miño":     {"mu":  700, "sigma": 200, "xi": 0.08, "n": 42},
    "Jucar":    {"mu":  420, "sigma": 130, "xi": 0.16, "n": 33},
}

am_data = {}
for name, p in stations.items():
    # genextreme uses -xi convention in scipy
    am_data[name] = genextreme.rvs(-p["xi"], loc=p["mu"], scale=p["sigma"],
                                    size=p["n"], random_state=rng)

# Quick overview
summary = pd.DataFrame({
    name: {"n": len(v), "mean": v.mean(), "std": v.std(),
           "max": v.max(), "min": v.min()}
    for name, v in am_data.items()
}).T.round(0)
print(summary)

                 n    mean    std     max    min
Ebro          40.0  1504.0  516.0  2932.0  833.0
Tajo          35.0  1003.0  238.0  1436.0  523.0
Duero         45.0  1280.0  380.0  2482.0  724.0
Guadalquivir  38.0   895.0  281.0  1693.0  498.0
Turia         30.0   482.0  224.0  1286.0  221.0
Segura        25.0   329.0  134.0   746.0  167.0
Miño          42.0   844.0  252.0  1474.0  436.0
Jucar         33.0   436.0  143.0   819.0  262.0


---
## 1. At-site fitting

Three estimation methods applied to a single station. In practice, use all three:
- L-moments: fast, always converges, recommended starting point
- MLE: more efficient (lower variance) for n ≥ 30, needed for likelihood-ratio tests
- Bayesian MCMC: full posterior; use for short records or when uncertainty quantification is critical

The comparison across methods reveals estimator stability — large disagreement between MLE and L-moments signals a problematic sample (outliers, poor convergence).

In [4]:
# Pick one station for the at-site demo
station_name = "Ebro"
data = am_data[station_name]

print(f"Station: {station_name}  |  n = {len(data)}  |  mean = {data.mean():.0f} m³/s")

Station: Ebro  |  n = 40  |  mean = 1504 m³/s


In [5]:
# --- MLE ---
params_mle = fit_gev_mle(data)
print("MLE parameters:")
print(f"  mu = {params_mle['mu']:.1f}  sigma = {params_mle['sigma']:.1f}  xi = {params_mle['xi']:.3f}")

MLE parameters:
  mu = 1244.6  sigma = 341.9  xi = 0.168


In [6]:
# --- L-moments (requires lmoments3) ---
try:
    params_lmom = fit_gev_lmom(data)
    print("L-moments parameters:")
    print(f"  mu = {params_lmom['mu']:.1f}  sigma = {params_lmom['sigma']:.1f}  xi = {params_lmom['xi']:.3f}")
except ImportError:
    params_lmom = params_mle   # fallback for the rest of the notebook
    print("(lmoments3 not installed — pip install lmoments3)")

L-moments parameters:
  mu = 1250.4  sigma = 367.7  xi = 0.103


In [7]:
# Bayesian MCMC — 2 chains / 500 samples for speed (use 4 / 1000 for production)
posterior = fit_gev_bayes(data, n_chains=2, n_samples=500)
print("Bayesian GEV posterior (Ebro station):")
print(posterior.describe().round(2))


Bayesian GEV posterior (Ebro station):
            mu    sigma       xi
count  1000.00  1000.00  1000.00
mean   1394.84   365.01     0.36
std     151.15    38.29     0.21
min    1051.94   223.72    -0.14
25%    1248.71   359.55     0.17
50%    1526.27   366.24     0.55
75%    1538.31   366.24     0.55
max    1538.31   558.18     0.65


Initializing NUTS using jitter+adapt_diag...
/usr/local/lib/python3.11/site-packages/pytensor/link/c/cmodule.py:2986: UserWarning: PyTensor could not link to a BLAS installation. Operations that might benefit from BLAS will be severely degraded.
This usually happens when PyTensor is installed via pip. We recommend it be installed via conda/mamba/pixi instead.
Alternatively, you can use an experimental backend such as Numba or JAX that perform their own BLAS optimizations, by setting `pytensor.config.mode == 'NUMBA'` or passing `mode='NUMBA'` when compiling a PyTensor function.
For more options and details see https://pytensor.readthedocs.io/en/latest/troubleshooting.html#how-do-i-configure-test-my-blas-library
  warnings.warn(
Multiprocess sampling (2 chains in 2 jobs)
NUTS: [mu_raw, sigma, xi]
                                                                                
                             Step      Grad                                     
  Progre…   Draw   Diverg…   siz

### Return level plot

The return level plot (Gumbel probability paper) shows:
- **Empirical points** (Gringorten plotting positions): observed data without any model assumption
- **Fitted curves**: model extrapolation into unobserved return periods
- **Uncertainty bands**: widen rapidly for T > record length — this is expected

> **Reading the plot:** Where the fitted curve passes through the empirical cloud is well-constrained. Where it extrapolates beyond the rightmost point is model-dependent — this is where different methods (MLE vs Bayes) differ most.

In [8]:
T_values = [2, 5, 10, 20, 50, 100, 200, 500, 1000]

rl_mle  = [return_level(params_mle,  T) for T in T_values]
rl_lmom = [return_level(params_lmom, T) for T in T_values]

# Empirical plotting position (Gringorten)
data_sorted = np.sort(data)
n = len(data_sorted)
prob = (np.arange(1, n + 1) - 0.44) / (n + 0.12)
T_emp = 1.0 / (1.0 - prob)

fig, ax = plt.subplots(figsize=(9, 5))
ax.semilogx(T_emp, data_sorted, 'ko', ms=4, label="Observed", zorder=4)
ax.semilogx(T_values, rl_mle,  'b-',  lw=2, label="GEV MLE")
ax.semilogx(T_values, rl_lmom, 'r--', lw=2, label="GEV L-moments")
ax.set_xlabel("Return period (years)")
ax.set_ylabel("Annual maximum discharge (m³/s)")
ax.set_title(f"Return level plot — {station_name}", fontsize=12)
ax.legend()
ax.grid(which="both", alpha=0.3)
plt.tight_layout()
plt.show()

# Table
rl_df = pd.DataFrame({"T": T_values, "MLE": np.round(rl_mle, 0), "L-mom": np.round(rl_lmom, 0)})
rl_df.set_index("T", inplace=True)
rl_df

### Bayesian return levels with credible intervals

When `fit_gev_bayes` is available, `return_level_bayes` extracts posterior
quantiles to build a credible interval around the return level curve.

In [9]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.semilogx(T_emp, data_sorted, "ko", ms=4, label="Observed", zorder=4)

# Return periods below 2 years are not useful for annual-maxima analysis
# (T=1 → p=0 hits the distribution boundary and produces extreme/non-finite values).
T_fine = np.logspace(np.log10(2), 3, 80)   # T from 2 to 1000 years

medians, lowers, uppers = [], [], []
for T in T_fine:
    ci = return_level_bayes(posterior, T, credible=0.90)
    medians.append(ci["median"])
    lowers.append(ci["lower"])
    uppers.append(ci["upper"])

ax.fill_between(T_fine, lowers, uppers, alpha=0.25, color="steelblue", label="90% CI")
ax.semilogx(T_fine, medians, "b-", lw=2, label="Posterior median")
ax.set_xlabel("Return period (years)")
ax.set_ylabel("Annual maximum discharge (m³/s)")
ax.set_title(f"Bayesian GEV — station {station_name}", fontsize=11)
ax.legend()
ax.grid(which="both", alpha=0.3)
plt.tight_layout()
plt.show()

---
## 2. Regional frequency analysis — Index Flood method

### Assumptions and pre-conditions

Before pooling, verify:
1. **Hydrological homogeneity**: stations should have similar flood generation mechanisms (same climate zone, similar basin characteristics)
2. **Discordancy**: no single station should have unusual statistics that distort the regional fit
3. **Record independence**: station records should not overlap in time (avoid double-counting shared events — check cross-correlations)

### Pooling mechanism

After normalising (dividing by the index flood), all station series should have mean ≈ 1.0. Fitting one GEV to the pooled data effectively uses `sum(n_i)` station-years instead of `n_i` from a single station — potentially a 5–10× increase in effective sample size.

In [10]:
_, index_floods = regional_index_flood(am_data)
print("Index floods (mean annual maximum, m³/s):")
for name, val in index_floods.items():
    print(f"  {name:<15}: {val:.0f}")

Index floods (mean annual maximum, m³/s):
  Ebro           : 1504
  Tajo           : 1003
  Duero          : 1280
  Guadalquivir   : 895
  Turia          : 482
  Segura         : 329
  Miño           : 844
  Jucar          : 436


In [11]:
# Fit regional GEV
regional_params, _ = fit_regional_gev(am_data, method="lmom")   # or 'mle'
print("\nRegional GEV parameters (L-moments):")
print(f"  mu = {regional_params['mu']:.3f}  sigma = {regional_params['sigma']:.3f}  xi = {regional_params['xi']:.3f}")
print("  (fitted to standardised series — all means ≈ 1.0)")


Regional GEV parameters (L-moments):
  mu = 0.843  sigma = 0.246  xi = 0.058
  (fitted to standardised series — all means ≈ 1.0)


In [12]:
# Regional return levels at all stations
T_values = [2, 5, 10, 25, 50, 100, 200, 500]
rl_regional = regional_return_levels(am_data, T_values=T_values, method="lmom")
print("Regional return levels (m³/s):")
rl_regional.round(0)

Regional return levels (m³/s):


In [13]:
# Compare at-site MLE vs regional L-mom for each station
station_names = list(am_data.keys())
T_ref = [10, 100]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, T in zip(axes, T_ref):
    atsite = [
        return_level(fit_gev_mle(am_data[n]), T)
        for n in station_names
    ]
    regional = rl_regional[f"T{T}"].values

    x = np.arange(len(station_names))
    ax.bar(x - 0.2, atsite,   width=0.35, label="At-site MLE",    color="steelblue", alpha=0.8)
    ax.bar(x + 0.2, regional, width=0.35, label="Regional L-mom", color="tomato",    alpha=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(station_names, rotation=35, ha="right", fontsize=9)
    ax.set_ylabel("Discharge (m³/s)")
    ax.set_title(f"T = {T} years", fontsize=11)
    ax.legend(fontsize=9)
    ax.grid(axis="y", alpha=0.3)

plt.suptitle("At-site MLE vs regional L-moments — return levels", fontsize=12)
plt.tight_layout()
plt.show()

---
## 3. Hosking & Wallis heterogeneity check

### L-coefficient of variation (L-CV)

The **L-CV** (= λ₂/λ₁) measures relative spread. In a perfectly homogeneous region, all stations would have the same L-CV. In practice, some variation is expected.

**Interpretation:**
- **Coefficient of variation of L-CV < 0.15**: region is approximately homogeneous — proceed with pooling
- **0.15 < CV < 0.30**: moderately heterogeneous — use pooling cautiously; investigate outlier stations
- **CV > 0.30**: strongly heterogeneous — do not pool; split into sub-regions or use hierarchical Bayes

### Why L-moments instead of regular moments?
L-moments are **linear combinations of order statistics**, making them resistant to outliers and more reliable than regular moments for small samples. The L-CV is a more stable measure of heterogeneity than the ordinary CV.

In [14]:
try:
    import lmoments3
    from lmoments3 import lmom_ratios

    lcv = {}
    for name, vals in am_data.items():
        lm = lmom_ratios(vals, nmom=4)
        lcv[name] = lm[1] / lm[0]  # L-CV = λ₂/λ₁

    lcv_series = pd.Series(lcv, name="L-CV")
    print(lcv_series.round(3))
    coeff_var = lcv_series.std() / lcv_series.mean()
    print(f"\nL-CV range: {lcv_series.min():.3f} – {lcv_series.max():.3f}")
    print(f"Coefficient of variation of L-CV: {coeff_var:.3f}")
    if coeff_var < 0.15:
        print("→ CoV < 0.15: region is approximately homogeneous")
    else:
        print("→ CoV ≥ 0.15: stations show significant L-CV spread.")
        print("  Note: this synthetic dataset has different σ/μ ratios by design,")
        print("  so the homogeneity test fails even though a common regional xi was used.")

    fig, ax = plt.subplots(figsize=(8, 3))
    lcv_series.plot(kind="bar", ax=ax, color="steelblue", alpha=0.8)
    ax.axhline(lcv_series.mean(), color="k", lw=1.5, ls="--", label="Regional mean")
    ax.set_ylabel("L-CV")
    ax.set_title("L-CV by station (homogeneity check)", fontsize=11)
    ax.legend()
    ax.grid(axis="y", alpha=0.3)
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()

except ImportError:
    print("(lmoments3 not installed — pip install lmoments3)")

Ebro            0.188
Tajo            0.140
Duero           0.162
Guadalquivir    0.172
Turia           0.249
Segura          0.218
Miño            0.171
Jucar           0.183
Name: L-CV, dtype: float64

L-CV range: 0.140 – 0.249
Coefficient of variation of L-CV: 0.184
→ CoV ≥ 0.15: stations show significant L-CV spread.
  Note: this synthetic dataset has different σ/μ ratios by design,
  so the homogeneity test fails even though a common regional xi was used.


---
## 4. Flood frequency map

Interpolate regional return levels onto a spatial grid using IDW to produce a
flood frequency map — combines `regional_return_levels` with
`IDWInterpolator`.

In [15]:
from pyhydra.climate.spatial_analysis import IDWInterpolator

# Approximate station coordinates (Iberian Peninsula)
station_coords = pd.DataFrame({
    "lon": [-0.88, -5.63, -5.10, -5.60, -0.39, -1.05, -7.86, -1.16],
    "lat": [41.66, 39.68, 41.55, 37.95, 39.46, 38.11, 42.60, 39.17],
}, index=station_names)

# Grid for Spain
glon2 = np.arange(-9.5, 4.5, 0.3)
glat2 = np.arange(35.5, 44.5, 0.3)
GLON2, GLAT2 = np.meshgrid(glon2, glat2)
grid2 = pd.DataFrame({"lon": GLON2.ravel(), "lat": GLAT2.ravel()})

T_map = 100
station_coords[f"T{T_map}"] = rl_regional[f"T{T_map}"].values

idw_rfa = IDWInterpolator(power=2)
idw_rfa.fit(station_coords, x_col="lon", y_col="lat", value_col=f"T{T_map}")
rl_map = idw_rfa.predict(grid2, x_col="lon", y_col="lat")

fig, ax = plt.subplots(figsize=(9, 6))
im = ax.pcolormesh(GLON2, GLAT2, rl_map.reshape(GLAT2.shape),
                   cmap="YlOrRd", shading="auto")
ax.scatter(station_coords.lon, station_coords.lat,
           c=station_coords[f"T{T_map}"], cmap="YlOrRd",
           edgecolors="k", s=80, zorder=3, linewidths=0.8)
for i, name in enumerate(station_names):
    ax.annotate(name, (station_coords.lon.iloc[i], station_coords.lat.iloc[i]),
                textcoords="offset points", xytext=(5, 3), fontsize=7)
plt.colorbar(im, ax=ax, label=f"Q_T{T_map} (m³/s)")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title(f"Regional T={T_map}-year flood frequency map", fontsize=12)
plt.tight_layout()
plt.show()